# Netflix Content-Based Recommender (TF-IDF + Cosine Similarity)

This notebook walks through the recommender end-to-end:

1. Load and explore the Netflix titles dataset (~7,000 titles).
2. Clean and combine textual metadata (title, plot, genres, cast, language, type).
3. Vectorize the corpus with TF-IDF (unigrams + bigrams, English stop-words removed).
4. Rank candidates by cosine similarity to a query title.
5. Use the packaged `NetflixRecommender` class for reusable inference.

## 1. Load the dataset

In [ ]:
import pandas as pd

movies = pd.read_csv('netflix_list.csv')
print(movies.shape)
movies.head(3)

## 2. Quick EDA

In [ ]:
print('columns:', list(movies.columns))
print('\ntypes:\n', movies['type'].value_counts())
print('\nmissing values per column:\n', movies.isna().sum().sort_values(ascending=False).head(10))

## 3. Drop irrelevant columns and build a content corpus

We drop identifiers and image URLs, fill NaN text fields, and concatenate the descriptive columns into a single `content` string per title.

In [ ]:
df = movies.drop(columns=['imdb_id', 'certificate', 'image_url']).copy()
df['title'] = df['title'].astype(str).str.strip()
df = df.dropna(subset=['title']).drop_duplicates(subset=['title']).reset_index(drop=True)

text_cols = ['title', 'plot', 'summary', 'genres', 'cast', 'language', 'type']
for col in text_cols:
    df[col] = df[col].fillna('').astype(str).str.strip()

df['content'] = df[text_cols].agg(' '.join, axis=1).str.lower()
df[['title', 'content']].head(2)

## 4. TF-IDF vectorization

We use unigrams + bigrams, remove English stop-words, and apply sublinear TF scaling so that high-frequency terms don't dominate the cosine similarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english',
    sublinear_tf=True,
)
movie_vectors = vectorizer.fit_transform(df['content'])
movie_vectors.shape

## 5. Cosine similarity matrix

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity = cosine_similarity(movie_vectors).astype(np.float32)
similarity.shape

## 6. Recommendation function

In [ ]:
title_to_index = {title.casefold(): i for i, title in enumerate(df['title'])}

def recommend(title, top_n=5):
    idx = title_to_index[title.casefold()]
    scores = similarity[idx]
    ranked = np.argsort(-scores)
    results = []
    for j in ranked:
        if j == idx:
            continue
        results.append((df.iloc[j]['title'], float(scores[j])))
        if len(results) >= top_n:
            break
    return results

for title, score in recommend('Narcos'):
    print(f'{score:0.3f}  {title}')

## 7. Use the packaged module

The same logic is exposed as a reusable class in `recommender.py` and as a CLI in `app.py`.

In [ ]:
from recommender import NetflixRecommender

rec = NetflixRecommender.from_csv('netflix_list.csv')
for r in rec.recommend('Stranger Things', top_n=5):
    print(f'{r.score:0.3f}  {r.title} ({r.year})  -  {r.genres}')